# Mamba: State Space Model - 실습 코드 1: Mamba 모델 사용 (mamba-ssm)

- Tutorial ID: `expand-mamba-ssm`
- Tutorial: Mamba: State Space Model
- Section ID: `expand-mamba-ssm-code-1`
- Section: 실습 코드 1: Mamba 모델 사용 (mamba-ssm)

이 노트북에서는 `mamba-ssm` 라이브러리가 제공하는 `Mamba` 블록을 직접 만들어보고, 입력 텐서를 흘려보내면서 shape과 파라미터 수를 하나하나 손으로 확인합니다. 수식을 유도하기보다는 "숫자가 실제로 어떻게 움직이는지"를 코드로 직접 보는 데 집중합니다.

**이 노트북에서 배우는 것**
- `Mamba` 블록을 구성하는 4개의 하이퍼파라미터(`d_model`, `d_state`, `d_conv`, `expand`)의 의미
- 입력 텐서 `(batch, seq_len, d_model)`이 Mamba를 통과해도 shape이 그대로 유지되는 이유
- 파라미터 수를 공식으로 직접 계산해서 실제 모델 값과 맞춰보기
- Attention(`∝ L²`)과 Mamba(`∝ L`)의 연산량 차이를 숫자로 비교하기

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Mamba 모델 사용 (mamba-ssm)
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 텐서 (batch, seq_len, d_model)가 Mamba 블록을 통과해도
#      shape이 그대로 유지된다는 것을 직접 확인
#   2) d_model / d_state / d_conv / expand 네 하이퍼파라미터가
#      모델 내부의 어떤 층을 결정하는지 이해
#   3) 파라미터 수를 "공식으로 직접 계산"해서 실제 모델 값과 맞춰보기
#   4) Attention(Transformer)과 비교했을 때 "선형 시간(linear time)"이
#      실제로 무슨 의미인지 숫자로 확인
#
# 읽는 순서:
#   1) 하이퍼파라미터(batch_size, seq_len, d_model, d_state, d_conv, expand)를
#      먼저 확인합니다.
#   2) 입력 텐서 x가 어떤 shape으로 만들어지는지 봅니다.
#   3) in_proj / conv1d / x_proj / dt_proj / A_log / D / out_proj 같은
#      내부 층이 각각 어떤 shape으로 투영하는지 확인합니다.
#   4) forward(순전파) 직후의 shape을 출력·assert로 검증합니다.
#   5) d_state, d_conv, expand 값을 바꿔가며 파라미터 수가
#      어떻게 달라지는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape이 그대로 유지되는가"와
#     "파라미터 수가 어느 하이퍼파라미터에서 늘어나는가"를 보세요.
#   - 이 노트북은 CUDA(GPU)가 있어야 실행됩니다. GPU가 없다면
#     Colab에서 [런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU]로
#     바꾼 뒤 실행하세요. (이유는 바로 아래 "1. 실행 환경 준비" 참고)

## 0. 들어가기 전에 — 이 노트북은 무엇을 실습하나요?

Transformer의 핵심 연산인 **Self-Attention**은 시퀀스 길이를 `L`이라고 할 때, "모든 토큰이 다른 모든 토큰을 한 번씩 서로 쳐다보는" 방식으로 동작합니다. 그래서 연산량이 `L²`(제곱)에 비례해서 늘어납니다. 문장이 2배 길어지면 연산량은 4배로 늘어나는 셈입니다.

**Mamba**는 이 문제를 해결하기 위해 등장한 State Space Model(SSM) 계열 아키텍처입니다. Mamba는 토큰을 처음부터 끝까지 순서대로 "한 번씩만" 훑으면서 상태(state)를 갱신하기 때문에, 연산량이 `L`(1제곱)에만 비례합니다. 그래서 아주 긴 시퀀스(예: DNA 서열, 오디오, 긴 문서)를 Transformer보다 훨씬 적은 계산으로 처리할 수 있습니다.

이 실습에서는 SSM의 수식을 유도하지 않고, `mamba-ssm` 라이브러리가 제공하는 `Mamba` 블록을 **레고 블록처럼 가져다 써보면서**, 아래 세 가지를 직접 손으로 확인합니다.

| 확인할 것 | 방법 |
|---|---|
| ① 입력/출력 shape이 그대로 유지되는가 | `x.shape`과 `y.shape`을 출력해서 비교 |
| ② 하이퍼파라미터가 파라미터 수를 어떻게 바꾸는가 | 공식으로 직접 계산해서 실제 값과 비교 |
| ③ 정말 "선형 시간"인가 | Attention과 연산량을 숫자로 비교 |

> 💡 `Mamba` 클래스는 Transformer의 "Multi-Head Attention" 층 하나를 대체하는 부품이라고 생각하면 됩니다. 실제 언어모델을 만들려면 이 블록을 residual connection + normalization과 함께 여러 층 쌓아야 하는데, 그 전체 모델(`MambaLMHeadModel`)은 다음 실습 코드에서 다룹니다.

## 1. 실행 환경 준비

`mamba-ssm`은 순수 파이썬 코드가 아니라 **CUDA로 직접 작성된 고속 커널**을 사용합니다. (SSM의 재귀적인 계산을 GPU에서 병렬로 빠르게 처리하는 "selective scan" 커널입니다.) 그래서 아래 조건이 반드시 필요합니다.

- **Linux** 운영체제
- **NVIDIA GPU** (CUDA 지원)
- **PyTorch** 1.12 이상 (CUDA 버전으로 설치된 것)
- **CUDA** 11.6 이상

macOS나 CPU 전용 환경에서는 설치는 되어도 실행 시 에러가 납니다. 로컬에 GPU가 없다면 Colab에서 **[런타임] → [런타임 유형 변경] → 하드웨어 가속기: GPU**로 바꾼 뒤 아래 셀을 실행하세요.

> 🔧 설치가 오래 걸리거나 실패한다면?
> - 설치가 너무 오래 걸린다 → 소스를 직접 컴파일하지 않도록 `--no-build-isolation` 옵션을 꼭 붙이세요. (이미 설치된 CUDA 버전 PyTorch를 그대로 사용하게 됩니다)
> - `causal-conv1d` 관련 에러가 난다 → `pip install causal-conv1d>=1.4.0` 을 따로 한 번 더 실행해보세요.

In [ ]:
# mamba-ssm 설치
#   - causal-conv1d: Mamba 블록 안에서 쓰이는 짧은 1D 합성곱(convolution)을
#     빠르게 계산해주는 보조 패키지입니다. (없어도 동작은 하지만 훨씬 느립니다)
#   - mamba-ssm: 이 실습에서 사용할 Mamba 블록 본체입니다.
#   - --no-build-isolation: 이미 설치되어 있는 CUDA 버전 PyTorch를 그대로
#     사용하라는 옵션입니다. (빠뜨리면 pip이 CPU 전용 PyTorch를 별도로
#     설치하려고 시도해서 설치가 꼬일 수 있습니다)
!pip install mamba-ssm[causal-conv1d] --no-build-isolation

import torch

# 실습을 시작하기 전에 GPU를 실제로 쓸 수 있는지 먼저 확인합니다.
# torch.cuda.is_available()이 False인 상태로 아래에서 .cuda()를 호출하면
# "Found no NVIDIA driver on your system" 같은 에러가 발생합니다.
if torch.cuda.is_available():
    print(f"✅ GPU 사용 가능: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU를 찾을 수 없습니다.")
    print("   Colab이라면 [런타임 > 런타임 유형 변경 > GPU]로 바꾼 뒤 다시 실행하세요.")

## 2. 하이퍼파라미터 이해하기

`Mamba` 블록을 만들 때 넣는 4개의 숫자는 각각 모델의 서로 다른 부분을 결정합니다. 코드를 실행하기 전에 하나씩 짚고 넘어갑니다.

| 이름 | 의미 | Transformer와 비교 |
|---|---|---|
| `d_model` | 토큰 하나를 표현하는 벡터의 차원 수 | Transformer의 `hidden_size`(임베딩 차원)와 동일한 역할 |
| `d_state` (N) | 시퀀스의 "과거 정보"를 요약해서 들고 다니는 은닉 상태(hidden state)의 크기 | RNN의 hidden state 크기와 비슷한 역할. 클수록 더 많은 과거 정보를 기억할 수 있음 |
| `d_conv` | SSM에 넣기 전, 바로 옆 토큰 몇 개를 미리 섞어주는 1D 합성곱의 커널 크기 | Attention이 "멀리 있는 토큰까지" 본다면, 이 합성곱은 "바로 옆 토큰(지역 정보)"을 먼저 섞는 전처리 역할 |
| `expand` (E) | 블록 내부 계산을 더 넓은 차원(`d_inner = expand × d_model`)에서 수행하고 다시 `d_model`로 되돌리는 확장 배수 | Transformer FFN이 `4 × d_model`로 확장했다가 되돌리는 것과 같은 아이디어 |

즉, 실제로 SSM 연산이 일어나는 내부 차원은 `d_model`이 아니라 `d_inner = expand × d_model`이라는 점이 중요합니다. 아래 코드에서 이 값들을 직접 변수로 만들어두고, 이후 모든 계산에서 그대로 재사용합니다.

In [ ]:
import math

# 하이퍼파라미터를 변수로 미리 선언해둡니다.
# (Mamba(...) 호출문 안에 숫자를 바로 적지 않고 변수로 분리해두면,
#  아래에서 파라미터 수를 "직접 계산"할 때 같은 값을 그대로 재사용할 수 있습니다)

d_model = 768   # 토큰 하나를 표현하는 벡터 차원 (BERT-base와 동일한 크기)
d_state = 16    # SSM 은닉 상태 크기 (N)
d_conv  = 4     # 로컬 1D 합성곱 커널 크기
expand  = 2     # 내부 확장 배수 (d_inner = expand * d_model)

batch_size = 2   # 한 번에 처리할 문장(시퀀스) 개수
seq_len    = 128 # 문장 하나에 들어있는 토큰(단어 조각) 개수

# 아래 두 값은 우리가 직접 정하는 값이 아니라, mamba-ssm 라이브러리
# 내부에서 d_model로부터 자동으로 계산되는 "파생값"입니다.
# (Mamba(...)의 dt_rank 인자 기본값이 "auto"이며, 이때 아래 식으로 계산됩니다)
d_inner = expand * d_model        # SSM이 실제로 연산을 수행하는 내부 차원
dt_rank = math.ceil(d_model / 16) # Δ(스텝 크기)를 만들 때 쓰는 저차원 랭크

print(f"d_model (입력/출력 차원)      : {d_model}")
print(f"d_inner (SSM 내부 차원)       : {d_inner}  (= expand({expand}) x d_model({d_model}))")
print(f"dt_rank (Δ의 랭크, 자동 계산) : {dt_rank}  (= ceil(d_model({d_model}) / 16))")
print(f"d_state (은닉 상태 크기)      : {d_state}")
print(f"d_conv  (합성곱 커널 크기)    : {d_conv}")

## 3. Mamba 블록 만들기

이제 위에서 정의한 하이퍼파라미터로 실제 `Mamba` 블록을 생성합니다. `Mamba(...)`는 내부적으로 아래 7개의 하위 층(sub-module)을 만듭니다. (지금은 이름과 역할만 훑어보고, 4번 섹션에서 실제로 하나씩 shape을 눈으로 확인합니다)

| 하위 층 | 역할 (한 문장 요약) |
|---|---|
| `in_proj` | 입력을 `x`(SSM에 들어갈 신호)와 `z`(게이트)로 나누기 위해 2배 차원으로 투영 |
| `conv1d` | 바로 옆 토큰들을 섞는 짧은 causal(미래를 보지 않는) 1D 합성곱 |
| `x_proj` | 각 토큰마다 "이번 토큰에 맞는" SSM 파라미터(Δ, B, C)를 만들어내는 투영 |
| `dt_proj` | 위에서 나온 Δ(스텝 크기)를 `d_inner` 차원으로 되돌리는 투영 |
| `A_log`, `D` | 모든 토큰이 공통으로 사용하는 SSM 고정 파라미터 |
| `out_proj` | SSM 결과를 다시 `d_model` 차원으로 되돌리는 투영 |

`x_proj`가 토큰마다 다른 Δ, B, C를 만들어낸다는 점이 바로 Mamba 논문 제목의 "Selective"(선택적) State Space Model이라는 이름의 핵심입니다. 모든 토큰에 똑같은 계산을 반복하는 게 아니라, **토큰 내용에 따라 SSM 파라미터 자체가 달라집니다.**

In [ ]:
from mamba_ssm import Mamba

# Mamba 블록 생성
# - 이 클래스 하나가 Transformer의 "Multi-Head Attention" 층 하나를
#   대체하는 부품입니다.
# - .cuda()를 붙여서 모델의 모든 파라미터를 GPU 메모리로 옮깁니다.
#   (mamba-ssm의 고속 커널은 GPU 텐서에서만 동작합니다)
model = Mamba(
    d_model=d_model,  # 모델 차원 (입력/출력 차원)
    d_state=d_state,  # SSM 상태 확장 차원 (N)
    d_conv=d_conv,    # 로컬 합성곱 폭
    expand=expand,    # 블록 확장 배수 (E)
).cuda()

# 층 구조를 그대로 출력해봅니다. 3번 섹션 표에 있던 이름들이
# (in_proj, conv1d, x_proj, dt_proj, out_proj) 그대로 보일 것입니다.
print(model)

## 4. 모델 내부 들여다보기

`print(model)`로 층 구조를 봤다면, 이번에는 각 층의 **shape**과 **파라미터 개수**를 직접 출력해서 3번 섹션 표와 실제로 맞춰봅니다. `named_parameters()`는 `(이름, 텐서)` 쌍을 순서대로 돌려주는 PyTorch의 기본 메서드입니다.

또한 `model.d_inner`, `model.dt_rank`처럼 Mamba 객체가 생성 시 자동으로 계산해서 들고 있는 파생값도 함께 확인해서, 우리가 2번 섹션에서 손으로 계산한 값과 실제로 같은지 검증합니다.

In [ ]:
# Mamba 객체는 생성 시 계산한 파생값들을 속성(attribute)으로도 가지고 있습니다.
# 우리가 2번 섹션에서 직접 계산한 d_inner, dt_rank와 같은 값이어야 합니다.
print(f"model.d_inner : {model.d_inner}  (우리가 계산한 값: {d_inner})")
print(f"model.dt_rank : {model.dt_rank}  (우리가 계산한 값: {dt_rank})")
assert model.d_inner == d_inner and model.dt_rank == dt_rank
print("✅ 라이브러리 내부 계산값과 우리가 손으로 계산한 값이 일치합니다.\n")

# named_parameters(): 모델 안의 모든 학습 가능한 파라미터를
# (이름, 텐서) 쌍으로 순서대로 돌려줍니다.
print(f"{'이름':22s} | {'shape':>18s} | {'개수':>12s}")
print("-" * 60)

running_total = 0
for name, param in model.named_parameters():
    running_total += param.numel()
    print(f"{name:22s} | {str(tuple(param.shape)):>18s} | {param.numel():>12,}")

print("-" * 60)
print(f"{'합계':22s} | {'':>18s} | {running_total:>12,}")

## 5. 입력 텐서 만들기

이제 모델에 넣을 입력을 만듭니다. Mamba(그리고 Transformer)는 입력을 **3차원 텐서** `(batch, seq_len, d_model)`로 받습니다. 각 축의 의미는 다음과 같습니다.

| 축 | 이름 | 의미 | 이 실습에서의 값 |
|---|---|---|---|
| 0번 축 | `batch` | 한 번에 처리하는 문장(샘플)의 개수 | 2개 문장 |
| 1번 축 | `seq_len` | 문장 하나를 이루는 토큰(단어 조각)의 개수 | 문장당 128개 토큰 |
| 2번 축 | `d_model` | 토큰 하나를 표현하는 벡터의 차원 | 토큰당 768차원 벡터 |

즉, `torch.randn(2, 128, 768)`은 "**서로 다른 문장 2개**가 있고, **각 문장은 128개의 토큰**으로 이루어져 있고, **각 토큰은 768개의 실수로 표현**되어 있다"는 뜻입니다. (실제 학습에서는 이 벡터가 랜덤이 아니라 토큰 임베딩 층을 통과한 값이지만, 여기서는 shape이 어떻게 흘러가는지 보는 것이 목적이므로 랜덤 값을 사용합니다)

In [ ]:
# 입력 텐서 생성: (batch_size, seq_len, d_model) = (2, 128, 768)
# torch.randn: 평균 0, 표준편차 1인 정규분포에서 무작위 값을 뽑아 채웁니다.
x = torch.randn(batch_size, seq_len, d_model).cuda()

print(f"입력 x의 shape         : {x.shape}")
print(f"  ㄴ batch (문장 개수)  : {x.shape[0]}")
print(f"  ㄴ seq_len (토큰 개수): {x.shape[1]}")
print(f"  ㄴ d_model (벡터 차원): {x.shape[2]}")

## 6. 순전파(forward pass) 실행하기

`model(x)`를 호출하면 내부적으로 아래 순서로 계산이 진행됩니다.

1. `in_proj` → 입력을 `x`(SSM 신호)와 `z`(게이트)로 분리
2. `conv1d`(causal) + SiLU 활성화 → 바로 옆 토큰 정보를 살짝 섞음
3. `x_proj` + `dt_proj` → 토큰마다 다른 Δ, B, C 파라미터 계산 (Selective!)
4. **selective scan** → 토큰을 처음부터 끝까지 순서대로 훑으며 은닉 상태를 갱신하고, `z`로 게이팅 (RNN처럼 "다음 상태 = f(이전 상태, 현재 입력)" 형태의 재귀 계산이지만, GPU에서 병렬로 빠르게 계산되도록 특수하게 구현되어 있습니다)
5. `out_proj` → 다시 `d_model` 차원으로 투영

핵심은 **입력 shape과 출력 shape이 완전히 같다**는 점입니다. 그래야 Transformer 블록을 이 블록으로 그대로 갈아끼울 수 있기 때문입니다. 아래에서 이 사실을 직접 assert(단언문)로 확인합니다.

In [ ]:
# 순전파 실행 (선형 시간!)
# torch.no_grad(): 지금은 학습이 아니라 shape 확인이 목적이므로
# 그래디언트 계산을 꺼서 메모리를 아끼고 속도를 높입니다.
with torch.no_grad():
    y = model(x)

print(f"입력  x: {x.shape}")
print(f"출력  y: {y.shape}")

# 입력과 출력의 shape이 완전히 같은지 직접 확인합니다.
# (다르면 AssertionError가 발생합니다 — 즉 이 줄이 통과했다는 것 자체가
#  "Mamba는 Attention의 drop-in replacement(그대로 교체 가능한 부품)이다"
#  라는 설계를 확인해주는 셈입니다)
assert x.shape == y.shape, "입력과 출력의 shape이 달라졌습니다!"
print("✅ 입력/출력 shape 일치 확인 — Attention 층을 그대로 대체할 수 있는 구조입니다.")

# 파라미터 수 (다음 섹션에서 자세히 다시 계산해봅니다)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")

## 7. 파라미터 수 세어보기 — 공식으로 직접 검증하기

모델 크기(=학습해야 할 파라미터 수)는 GPU 메모리 사용량과 학습 속도에 직결되므로, 실무에서 항상 확인하는 값입니다. 두 가지 방법으로 같은 값을 구해서 서로 맞춰봅니다.

- **방법 A (실측)**: `model.parameters()`를 순회하며 `numel()`(원소 개수)을 모두 더함
- **방법 B (공식)**: 3번 섹션에서 본 7개 하위 층의 shape을 하이퍼파라미터 식으로 직접 계산해서 더함

두 값이 정확히 같아야 합니다. 이 노트북의 하이퍼파라미터(`d_model=768, d_state=16, d_conv=4, expand=2`) 기준으로는 두 값 모두 **3,770,880**이 나와야 합니다.

In [ ]:
# ── 방법 A: 실측 — 모델에 있는 모든 파라미터의 원소 개수를 더합니다.
total_actual = sum(p.numel() for p in model.parameters())

# ── 방법 B: 공식 — 3번 섹션 표에 나온 7개 하위 층을 하이퍼파라미터로 직접 계산합니다.
#    (mamba_ssm 소스코드 기본값 기준: in_proj / x_proj / out_proj는 bias 없음,
#     conv1d / dt_proj는 bias 있음)
p_in_proj  = d_model * (d_inner * 2)                # bias 없음
p_conv1d   = d_inner * d_conv + d_inner             # depthwise weight + bias
p_x_proj   = d_inner * (dt_rank + d_state * 2)      # bias 없음
p_dt_proj  = dt_rank * d_inner + d_inner            # weight + bias
p_A_log    = d_inner * d_state
p_D        = d_inner
p_out_proj = d_inner * d_model                      # bias 없음

total_formula = p_in_proj + p_conv1d + p_x_proj + p_dt_proj + p_A_log + p_D + p_out_proj

print(f"{'하위 층':10s} | {'파라미터 수':>12s}")
print("-" * 30)
print(f"{'in_proj':10s} | {p_in_proj:>12,}")
print(f"{'conv1d':10s} | {p_conv1d:>12,}")
print(f"{'x_proj':10s} | {p_x_proj:>12,}")
print(f"{'dt_proj':10s} | {p_dt_proj:>12,}")
print(f"{'A_log':10s} | {p_A_log:>12,}")
print(f"{'D':10s} | {p_D:>12,}")
print(f"{'out_proj':10s} | {p_out_proj:>12,}")
print("-" * 30)
print(f"공식으로 계산한 총합  : {total_formula:,}")
print(f"실제 모델 파라미터 수 : {total_actual:,}")

assert total_formula == total_actual, "공식과 실측값이 다릅니다! 라이브러리 버전에 따라 구조가 바뀌었을 수 있습니다."
print("✅ 공식으로 계산한 값과 실제 모델의 파라미터 수가 일치합니다.")

## 8. 어림식과 비교, 그리고 "왜 선형 시간인가"

### 8-1. 어림식(rule of thumb)과 비교

Mamba 공식 저장소(state-spaces/mamba) 예제 코드에는 이런 주석이 달려 있습니다.

> `Mamba` 블록은 대략 `3 × expand × d_model²`개의 파라미터를 사용한다

이 어림식은 7개 하위 층 중 **가장 큰 두 층인 `in_proj`와 `out_proj`만** 계산한 값입니다. `in_proj = 2 × expand × d_model²`, `out_proj = expand × d_model²`이므로 둘을 더하면 정확히 `3 × expand × d_model²`이 됩니다. `d_state`, `d_conv`, `dt_rank`는 `d_model`보다 훨씬 작은 값이라서, 이들에 비례하는 나머지 5개 층은 상대적으로 무시할 만큼 작습니다. 아래 코드에서 실제로 두 층이 전체의 몇 %를 차지하는지 계산해봅니다.

### 8-2. Attention과 연산량 비교

Self-Attention은 모든 토큰 쌍(pair) 사이의 관계를 계산하므로, 그 핵심 연산량이 **`시퀀스 길이(L)의 제곱`**에 비례합니다 (`∝ L² · d`). 반면 Mamba의 selective scan은 토큰을 한 번씩만 순서대로 훑으므로 연산량이 **`L`에 비례**합니다 (`∝ L · d`). (두 아키텍처에 공통으로 존재하는 `in_proj`/`out_proj` 같은 선형 투영 비용은 제외하고, "토큰들을 서로 섞는 핵심 연산"만 비교한 것입니다) 다음 셀에서 시퀀스 길이를 늘려가며 두 값의 비율이 어떻게 벌어지는지 직접 숫자로 확인합니다.

In [ ]:
# 어림식: 3 * expand * d_model^2  (in_proj + out_proj 두 층만 계산한 값)
quick_estimate = 3 * expand * d_model ** 2
big_two = p_in_proj + p_out_proj

print(f"어림식 (3 x expand x d_model^2) : {quick_estimate:,}")
print(f"in_proj + out_proj 실제 합      : {big_two:,}")
print(f"두 값이 같은가?                 : {quick_estimate == big_two}")
print()
print(f"in_proj + out_proj가 전체 파라미터에서 차지하는 비율 : {big_two / total_actual:.1%}")
print(f"어림식과 정확한 총합의 차이(%)                       : {(total_actual - quick_estimate) / total_actual:.1%}")

In [ ]:
# Attention : 연산량 ∝ L^2 * d   (모든 토큰 쌍을 계산)
# Mamba(SSM): 연산량 ∝ L   * d   (토큰을 순서대로 한 번만 훑음)
#
# 실제 FLOPs 상수항까지 정확히 맞춘 벤치마크는 아니고,
# "L에 대해 제곱으로 느는가 vs 1제곱으로 느는가"라는 핵심 차이를
# 눈으로 확인하기 위한 단순화된 비교입니다.

seq_lengths = [128, 1024, 8192, 65536]  # 128: 이 노트북에서 쓴 길이 / 65536: 매우 긴 문서(책 한 권 분량)

print(f"{'시퀀스 길이 L':>12} | {'Attention 연산량 (∝L²d)':>24} | {'Mamba 연산량 (∝Ld)':>18} | {'Attention이 몇 배 더 무거운가':>16}")
print("-" * 90)
for L in seq_lengths:
    attn_cost = L ** 2 * d_model
    mamba_cost = L * d_model
    ratio = attn_cost / mamba_cost
    print(f"{L:>12,} | {attn_cost:>24,} | {mamba_cost:>18,} | {ratio:>14,.0f}배")

print()
print("→ Attention이 Mamba보다 '몇 배' 무거워지는지는 시퀀스 길이 L 자신과 같아집니다.")
print("  (L^2*d / L*d = L 이기 때문입니다)")
print("  L=128에서는 128배 차이지만, L=65536에서는 65536배 차이로 벌어집니다.")
print("  이것이 '선형 시간(linear time)'이 긴 시퀀스에서 실전으로 의미를 갖는 지점입니다.")

## 9. 직접 실험해보기 (Try it yourself)

지금까지 만든 공식을 그대로 재사용해서, 하이퍼파라미터를 바꿔보며 "어디가 늘어나는지" 직접 확인해봅니다. 아래 함수는 7번 섹션에서 한 계산을 하나로 묶어, `d_model`/`d_state`/`d_conv`/`expand`만 바꿔서 바로 결과를 볼 수 있게 만든 것입니다.

**해보면 좋은 실험**
- `d_state`를 16 → 64로 늘리면 파라미터가 얼마나 늘어날까요? (힌트: `x_proj`, `A_log`에만 영향을 줍니다)
- `expand`를 2 → 4로 늘리면 파라미터가 얼마나 늘어날까요? (힌트: `d_inner`가 커지므로 **거의 모든** 층에 영향을 줍니다)
- `d_conv`를 4 → 8로 늘리면 파라미터가 얼마나 늘어날까요? (힌트: 다른 값들에 비해 영향이 거의 없습니다. 8-1에서 본 어림식을 떠올려 보세요)

In [ ]:
def count_mamba_params(d_model, d_state, d_conv, expand):
    '''
    실제 Mamba(...) 객체를 만들지 않고도, 하이퍼파라미터만으로
    파라미터 수를 예측하는 함수입니다. (7번 섹션의 공식을 함수로 정리한 것)
    '''
    d_inner = expand * d_model
    dt_rank = math.ceil(d_model / 16)

    p_in_proj  = d_model * (d_inner * 2)
    p_conv1d   = d_inner * d_conv + d_inner
    p_x_proj   = d_inner * (dt_rank + d_state * 2)
    p_dt_proj  = dt_rank * d_inner + d_inner
    p_A_log    = d_inner * d_state
    p_D        = d_inner
    p_out_proj = d_inner * d_model

    return p_in_proj + p_conv1d + p_x_proj + p_dt_proj + p_A_log + p_D + p_out_proj


# 기준값 (지금까지 이 노트북에서 사용한 설정)
base = count_mamba_params(d_model=768, d_state=16, d_conv=4, expand=2)

# 실험 1: d_state만 16 -> 64로
exp_d_state = count_mamba_params(d_model=768, d_state=64, d_conv=4, expand=2)

# 실험 2: expand만 2 -> 4로
exp_expand = count_mamba_params(d_model=768, d_state=16, d_conv=4, expand=4)

# 실험 3: d_conv만 4 -> 8로
exp_d_conv = count_mamba_params(d_model=768, d_state=16, d_conv=8, expand=2)

print(f"{'설정':22s} | {'파라미터 수':>14s} | {'기준 대비 증가율':>10s}")
print("-" * 55)
print(f"{'기준 (16, 4, 2)':22s} | {base:>14,} | {'-':>10s}")
print(f"{'d_state: 16 -> 64':22s} | {exp_d_state:>14,} | {(exp_d_state / base - 1):>9.1%}")
print(f"{'expand : 2 -> 4':22s} | {exp_expand:>14,} | {(exp_expand / base - 1):>9.1%}")
print(f"{'d_conv : 4 -> 8':22s} | {exp_d_conv:>14,} | {(exp_d_conv / base - 1):>9.1%}")

# 👉 아래 빈 칸을 직접 채워서 나만의 설정도 실험해보세요.
# my_params = count_mamba_params(d_model=768, d_state=32, d_conv=4, expand=3)
# print(f"내가 만든 설정: {my_params:,}개 (기준 대비 {my_params / base - 1:.1%})")

## 정리 (요약)

- `Mamba(d_model, d_state, d_conv, expand)`는 Transformer의 Attention 층 하나를 대체할 수 있는 부품이며, **입력과 출력의 shape이 항상 같습니다.**
- 내부적으로 `in_proj → conv1d(causal) → (x_proj, dt_proj로 토큰마다 다른 SSM 파라미터 생성) → selective scan → out_proj` 순서로 계산됩니다.
- 파라미터 수는 `in_proj`와 `out_proj`(둘을 합치면 약 `3 × expand × d_model²`)가 대부분을 차지하고, `d_state`/`d_conv`는 상대적으로 적은 파라미터만 추가하면서 "과거를 얼마나 기억할지" / "주변을 얼마나 섞을지"를 조절합니다.
- Attention은 핵심 연산량이 `L²`에 비례하지만 Mamba는 `L`에 비례하기 때문에, 시퀀스가 길어질수록 그 격차가 시퀀스 길이 자신만큼 벌어집니다.

## 다음 실습

이 노트북에서는 `Mamba` 블록 하나만 다뤘습니다. 다음 실습 코드에서는 이 블록을 residual connection + normalization과 함께 여러 층 쌓아 만든 전체 언어모델 `MambaLMHeadModel`을 다루고, 실제로 텍스트를 생성해봅니다.

## 참고 자료

- 논문: Gu & Dao, *Mamba: Linear-Time Sequence Modeling with Selective State Spaces* — https://arxiv.org/abs/2312.00752
- 공식 저장소: state-spaces/mamba — https://github.com/state-spaces/mamba